# TasteBuds — Food Delivery Analytics (pandas)

Twenty business questions about a food-delivery marketplace — revenue, retention,
cohorts, service quality and data integrity — answered on four related tables:
**couriers**, **customers**, **restaurants** and **orders** (420 orders, Jan–Jun 2025).

Each question below is answered in pandas and mirrors the matching query in
**`tastebuds-food-delivery-analysis.sql`**. The final section puts the same twenty
questions back through DuckDB — reading the same CSVs, not the pandas frames —
and asserts the two engines return identical answers.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# The four CSVs live next to this notebook.
DATA_DIR = Path.cwd()

Load the four tables and parse the two date columns once, so every downstream
`.dt` accessor and period grouping works without re-parsing.

In [2]:
couriers    = pd.read_csv(DATA_DIR / 'couriers.csv')
customers   = pd.read_csv(DATA_DIR / 'customers.csv',   parse_dates=['signup_date'])
restaurants = pd.read_csv(DATA_DIR / 'restaurants.csv')
orders      = pd.read_csv(DATA_DIR / 'orders.csv',      parse_dates=['order_date'])

orders.head()

,order_id,customer_id,restaurant_id,courier_id,order_date,order_total,status,delivery_minutes,rating
0,9001,30,102,503,2025-06-18,69.24,delivered,56.00,5.00
1,9002,42,112,508,2025-02-08,57.17,delivered,27.00,5.00
2,9003,107,112,507,2025-06-29,22.17,delivered,51.00,3.00
3,9004,73,114,501,2025-06-25,77.18,delivered,61.00,1.00
4,9005,83,104,502,2025-01-08,58.51,cancelled,NaN,NaN


Shape of each table, and the schema of the fact table.

In [3]:
for name, df in [('couriers', couriers), ('customers', customers),
                 ('restaurants', restaurants), ('orders', orders)]:
    print(f'{name:<12} {df.shape[0]:>4} rows x {df.shape[1]} cols')

orders.info()

couriers        8 rows x 4 cols
customers     120 rows x 4 cols
restaurants    15 rows x 5 cols
orders        420 rows x 9 cols
<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          420 non-null    int64         
 1   customer_id       420 non-null    int64         
 2   restaurant_id     420 non-null    int64         
 3   courier_id        420 non-null    int64         
 4   order_date        420 non-null    datetime64[us]
 5   order_total       420 non-null    float64       
 6   status            420 non-null    str           
 7   delivery_minutes  341 non-null    float64       
 8   rating            341 non-null    float64       
dtypes: datetime64[us](1), float64(3), int64(4), str(1)
memory usage: 29.7 KB


## Conventions

Three decisions are made once here and held for every question below.

1. **Revenue means delivered revenue.** Cancelled orders carry an `order_total`
   but never became money, so `delivered` — not `orders` — is the base for every
   revenue, AOV and spend figure.
2. **"City" means the customer's city.** `customers`, `couriers` and `restaurants`
   each have their own `city`; they answer different questions and give different
   splits. Demand-side questions use the customer's city throughout.
3. **`rating` means the rating left on the order.** `restaurants` also has a
   `rating` column (the restaurant's own listing score), so the restaurant merge
   passes explicit suffixes and the listing score arrives as `rating_restaurant` —
   an unsuffixed merge would leave `rating_x` / `rating_y` and quietly average
   the wrong one.

`delivery_minutes` and `rating` are NULL on every cancelled order, which is why
service-quality averages are taken over `delivered` only.

In [4]:
delivered = orders[orders['status'] == 'delivered'].copy()

by_customer   = delivered.merge(customers,   on='customer_id')
by_courier    = delivered.merge(couriers,    on='courier_id')
by_restaurant = delivered.merge(restaurants, on='restaurant_id',
                                suffixes=('', '_restaurant'))

print(f'{len(delivered)} delivered of {len(orders)} orders')
print('restaurant merge keeps both ratings:',
      [c for c in by_restaurant.columns if 'rating' in c])

341 delivered of 420 orders
restaurant merge keeps both ratings: ['rating', 'rating_restaurant']


## 1 — Revenue & order health

**Q1 — Total revenue (GMV) from delivered orders.**
`SQL: SELECT ROUND(SUM(order_total), 2) FROM orders WHERE status = 'delivered'`

In [5]:
gmv = delivered['order_total'].sum()
round(gmv, 2)

np.float64(16413.67)

**Q2 — How many orders in total, and how many were delivered?**
`SQL: COUNT(*), COUNT(*) FILTER (WHERE status = 'delivered')`

In [6]:
orders['status'].value_counts().rename('orders').to_frame()

,orders
status,
delivered,341
cancelled,79


**Q3 — Average order value (AOV) of delivered orders.**
`SQL: AVG(order_total) WHERE status = 'delivered'`

The original draft averaged *all* orders, including cancelled ones — the filter
is what makes this AOV rather than "average basket someone once assembled".

In [7]:
round(delivered['order_total'].mean(), 2)

np.float64(48.13)

**Q4 — Delivered revenue by restaurant cuisine, highest first.**
`SQL: JOIN restaurants USING (restaurant_id) GROUP BY cuisine ORDER BY revenue DESC`

In [8]:
revenue_by_cuisine = (
    by_restaurant.groupby('cuisine')
    .agg(orders=('order_id', 'count'),
         revenue=('order_total', 'sum'),
         avg_order_value=('order_total', 'mean'))
    .sort_values('revenue', ascending=False)
    .round(2)
)
revenue_by_cuisine

,orders,revenue,avg_order_value
cuisine,,,
Chinese,109,"5,365.53",49.23
Thai,85,"4,126.08",48.54
Italian,74,"3,453.90",46.67
American,24,"1,165.67",48.57
Mexican,24,"1,157.63",48.23
Indian,25,"1,144.86",45.79


**Q5 — Which five restaurants generated the most delivered revenue?**
`SQL: GROUP BY r.restaurant_id, r.name, r.cuisine ORDER BY revenue DESC LIMIT 5`

Grouped by the key rather than the label, so two restaurants sharing a name
could never collapse into one row.

In [9]:
top_restaurants = (
    by_restaurant.groupby(['restaurant_id', 'name', 'cuisine'], as_index=False)
    .agg(orders=('order_id', 'count'), revenue=('order_total', 'sum'))
    .sort_values('revenue', ascending=False)
    .head(5)
    .round(2)
)
top_restaurants[['name', 'cuisine', 'orders', 'revenue']]

,name,cuisine,orders,revenue
3,Spice Route,Chinese,27,"1,455.54"
14,Trattoria Sole,Chinese,27,"1,323.69"
10,Curry Leaf,Thai,26,"1,232.80"
9,Dragon Pearl,Italian,25,"1,220.95"
2,Golden Wok,American,24,"1,165.67"


**Q6 — What share of all orders were cancelled?**
`SQL: 100.0 * COUNT(*) FILTER (WHERE status = 'cancelled') / COUNT(*)`

A cancellation rate counts *orders*, not dollars — a value-weighted version
answers a different question and lands 0.3 pts higher here.

In [10]:
cancelled = orders['status'] == 'cancelled'
print(f'{cancelled.sum()} of {len(orders)} orders cancelled')
round(100 * cancelled.mean(), 2)

79 of 420 orders cancelled


np.float64(18.81)

**Q7 — Each city's delivered revenue as a share of the total.**
`SQL: SUM(order_total) / SUM(SUM(order_total)) OVER ()`

In [11]:
city_revenue = (
    by_customer.groupby('city')
    .agg(orders=('order_id', 'count'), revenue=('order_total', 'sum'))
    .sort_values('revenue', ascending=False)
)
city_revenue['pct_of_revenue'] = (
    100 * city_revenue['revenue'] / city_revenue['revenue'].sum()
).round(1)
city_revenue.round(2)

,orders,revenue,pct_of_revenue
city,,,
New York,102,"5,061.92",30.80
Chicago,69,"3,464.74",21.10
Austin,74,"3,462.14",21.10
Seattle,56,"2,767.96",16.90
Miami,40,"1,656.91",10.10


## 2 — Customer base & retention

**Q8 — Active customers, and revenue per active customer (ARPU).**
`SQL: COUNT(DISTINCT customer_id), SUM(order_total) / COUNT(DISTINCT customer_id)`

In [12]:
active_customers = delivered['customer_id'].nunique()
arpu = delivered['order_total'].sum() / active_customers

print(f'{active_customers} of {len(customers)} registered customers are active')
round(arpu, 2)

113 of 120 registered customers are active


np.float64(145.25)

**Q9 — What share of active customers placed more than one delivered order?**
`SQL: WITH per_customer AS (... GROUP BY customer_id) SELECT ... FILTER (WHERE delivered_orders > 1)`

The denominator is *active* customers, not all orders — a repeat rate is a
property of people, so the aggregation has to land on `customer_id` first.

In [13]:
orders_per_customer = delivered.groupby('customer_id').size()
repeat = orders_per_customer > 1

print(f'{repeat.sum()} of {len(orders_per_customer)} active customers ordered more than once')
round(100 * repeat.mean(), 1)

93 of 113 active customers ordered more than once


np.float64(82.3)

**Q10 — How many registered customers have never had an order delivered?**
`SQL: WHERE NOT EXISTS (SELECT 1 FROM orders WHERE ... AND status = 'delivered')`

The anti-join has to be built from *delivered* customer ids. Comparing against
every id in `orders` counts a customer whose only order was cancelled as active.

In [14]:
active_ids = set(delivered['customer_id'])
never_ordered = (~customers['customer_id'].isin(active_ids)).sum()

print(f'{never_ordered} of {len(customers)} registered customers never received an order')
never_ordered

7 of 120 registered customers never received an order


np.int64(7)

## 3 — Time series & window functions

**Q11 — Delivered revenue per month, with the month-over-month change.**
`SQL: LAG(revenue) OVER (ORDER BY month)`

Grouping on `dt.to_period('M')` rather than `dt.month` keeps the year in the key —
`dt.month` alone would fold January 2024 and January 2025 into one bucket.

In [15]:
monthly = (
    delivered.groupby(delivered['order_date'].dt.to_period('M'))
    .agg(revenue=('order_total', 'sum'))
)
monthly.index.name = 'month'

monthly['prev_month'] = monthly['revenue'].shift(1)      # SQL: LAG(...)
monthly['mom_change'] = monthly['revenue'].diff()
monthly['mom_pct']    = monthly['revenue'].pct_change() * 100

monthly.round(2)

,revenue,prev_month,mom_change,mom_pct
month,,,,
2025-01,"2,564.13",NaN,NaN,NaN
2025-02,"2,098.74","2,564.13",-465.39,-18.15
2025-03,"3,197.22","2,098.74","1,098.48",52.34
2025-04,"3,147.06","3,197.22",-50.16,-1.57
2025-05,"2,648.16","3,147.06",-498.90,-15.85
2025-06,"2,758.36","2,648.16",110.20,4.16


**Q12 — Cumulative (running-total) delivered revenue by month.**
`SQL: SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`

In [16]:
monthly['cumulative_revenue'] = monthly['revenue'].cumsum()
monthly[['revenue', 'cumulative_revenue']].round(2)

,revenue,cumulative_revenue
month,,
2025-01,"2,564.13","2,564.13"
2025-02,"2,098.74","4,662.87"
2025-03,"3,197.22","7,860.09"
2025-04,"3,147.06","11,007.15"
2025-05,"2,648.16","13,655.31"
2025-06,"2,758.36","16,413.67"


## 4 — Service quality

**Q13 — Average delivery time (minutes) by city.**
`SQL: AVG(delivery_minutes) ... WHERE status = 'delivered' GROUP BY c.city`

In [17]:
delivery_by_city = (
    by_customer.groupby('city')
    .agg(deliveries=('order_id', 'count'),
         avg_delivery_minutes=('delivery_minutes', 'mean'),
         slowest_delivery=('delivery_minutes', 'max'))
    .sort_values('avg_delivery_minutes')
    .round(2)
)
delivery_by_city

,deliveries,avg_delivery_minutes,slowest_delivery
city,,,
Miami,40,42.00,72.00
Chicago,69,45.00,73.00
Austin,74,45.07,75.00
New York,102,45.48,75.00
Seattle,56,46.39,75.00


**Q14 — Average customer rating by cuisine (rated orders only).**
`SQL: AVG(o.rating) ... WHERE o.rating IS NOT NULL GROUP BY r.cuisine`

`rating` is what the customer scored *this order*; `rating_restaurant` is the
restaurant's own listing score. Averaging the latter by cuisine answers a
different question and reorders the table completely.

In [18]:
rating_by_cuisine = (
    by_restaurant.groupby('cuisine')
    .agg(rated_orders=('rating', 'count'), avg_rating=('rating', 'mean'))
    .sort_values('avg_rating', ascending=False)
    .round(2)
)
rating_by_cuisine

,rated_orders,avg_rating
cuisine,,
Indian,25,4.16
Thai,85,4.09
Chinese,109,4.02
Italian,74,4.01
Mexican,24,3.83
American,24,3.79


## 5 — Ranking & per-group comparisons

**Q15 — Which five customers spent the most on delivered orders?**
`SQL: GROUP BY c.customer_id, c.name, c.city ORDER BY spend DESC LIMIT 5`

In [19]:
top_customers = (
    by_customer.groupby(['customer_id', 'name', 'city'], as_index=False)
    .agg(delivered_orders=('order_id', 'count'), spend=('order_total', 'sum'))
    .sort_values('spend', ascending=False)
    .head(5)
    .round(2)
)
top_customers

,customer_id,name,city,delivered_orders,spend
30,32,Xander Ross,Austin,8,371.78
22,24,Xander Cole,New York,6,367.23
32,34,Wren Chen,New York,7,363.30
75,81,Omar Osei,Chicago,7,319.77
68,72,Sara Shah,New York,6,303.51


Why the group key is `customer_id` and not `name`: the roster holds 120 customers
but only 103 distinct names, so grouping by name silently welds different people
together. Two unrelated customers are both called *Xander Ross* — grouped by name
they present as a single 545.86 spender who does not exist.

In [20]:
print(customers['name'].nunique(), 'distinct names across', len(customers), 'customers')

(by_customer.groupby('name')['order_total'].sum()
 .sort_values(ascending=False)
 .head(3)
 .round(2)
 .rename('spend_grouped_by_name'))

103 distinct names across 120 customers


name
Xander Ross   545.86
Zoe Shah      476.77
Isla Vale     465.82
Name: spend_grouped_by_name, dtype: float64

**Q16 — Rank cities by delivered revenue (1 = highest), handling ties.**
`SQL: RANK() OVER (ORDER BY revenue DESC)` → `rank(method='min')`;
`DENSE_RANK()` → `rank(method='dense')`.

In [21]:
city_rank = city_revenue[['revenue']].copy()
city_rank['revenue_rank'] = city_rank['revenue'].rank(method='min',   ascending=False).astype(int)
city_rank['dense_rank']   = city_rank['revenue'].rank(method='dense', ascending=False).astype(int)

city_rank.sort_values('revenue_rank').round(2)

,revenue,revenue_rank,dense_rank
city,,,
New York,"5,061.92",1,1
Chicago,"3,464.74",2,2
Austin,"3,462.14",3,3
Seattle,"2,767.96",4,4
Miami,"1,656.91",5,5


**Q17 — How many delivered orders are larger than that customer's own average?**
`SQL: AVG(order_total) OVER (PARTITION BY customer_id)` → `groupby(...).transform('mean')`

`transform` is the pandas analogue of a partitioned window: it returns one value
per *row* rather than one per group, so the comparison stays at order grain.

In [22]:
flagged = delivered.copy()
flagged['customer_aov'] = flagged.groupby('customer_id')['order_total'].transform('mean')

above_own_aov = flagged['order_total'] > flagged['customer_aov']
print(f'{above_own_aov.sum()} of {len(flagged)} delivered orders beat their own customer AOV')
round(100 * above_own_aov.mean(), 1)

159 of 341 delivered orders beat their own customer AOV


np.float64(46.6)

**Q18 — How many customers signed up in each month (cohort sizes)?**
`SQL: DATE_TRUNC('month', signup_date) GROUP BY 1`

In [23]:
cohorts = (
    customers.groupby(customers['signup_date'].dt.to_period('M'))
    .size()
    .rename('customers')
    .to_frame()
)
cohorts.index.name = 'signup_month'
cohorts

,customers
signup_month,
2024-09,30
2024-10,23
2024-11,18
2024-12,20
2025-01,29


**Q19 — Deliveries and average rating per courier; who are the busiest?**
`SQL: JOIN couriers USING (courier_id) GROUP BY co.courier_id, ... ORDER BY deliveries DESC`

In [24]:
courier_stats = (
    by_courier.groupby(['courier_id', 'name', 'city', 'vehicle'], as_index=False)
    .agg(deliveries=('order_id', 'count'), avg_rating=('rating', 'mean'))
    .sort_values('deliveries', ascending=False)
    .round(2)
)
courier_stats[['name', 'city', 'vehicle', 'deliveries', 'avg_rating']]

,name,city,vehicle,deliveries,avg_rating
1,Leo Shah,New York,Scooter,46,3.85
3,Ravi Cole,Chicago,Bike,46,4.22
7,Ava Diaz,Austin,Car,46,4.26
0,Ben Nowak,Austin,Bike,45,4.00
2,Mona Reyes,Seattle,Scooter,42,3.88
5,Isla Park,New York,Bike,41,3.80
4,Ravi Osei,New York,Bike,39,3.97
6,Mona Iyer,New York,Scooter,36,4.14


## 6 — Data-quality checks

**Q20 — Surface logically inconsistent rows.** Every check should return 0.

Each rule states an invariant the schema cannot enforce on its own: a cancelled
order should carry no delivery time or rating, a delivered one should carry both,
totals should be positive, ratings should sit on the 1–5 scale, and every foreign
key should resolve.

In [25]:
quality_checks = {
    'cancelled order still has a delivery time or rating':
        (orders['status'] == 'cancelled')
        & (orders['delivery_minutes'].notna() | orders['rating'].notna()),
    'delivered order missing a rating':
        (orders['status'] == 'delivered') & orders['rating'].isna(),
    'delivered order missing a delivery time':
        (orders['status'] == 'delivered') & orders['delivery_minutes'].isna(),
    'order_total is not positive':
        orders['order_total'] <= 0,
    'rating outside the 1-5 scale':
        orders['rating'].notna() & ~orders['rating'].between(1, 5),
    'status outside the known vocabulary':
        ~orders['status'].isin(['delivered', 'cancelled']),
    'duplicate order_id':
        orders['order_id'].duplicated(keep=False),
    'order references a customer that does not exist':
        ~orders['customer_id'].isin(customers['customer_id']),
    'order references a restaurant that does not exist':
        ~orders['restaurant_id'].isin(restaurants['restaurant_id']),
    'order references a courier that does not exist':
        ~orders['courier_id'].isin(couriers['courier_id']),
}

quality = (
    pd.Series({name: int(mask.sum()) for name, mask in quality_checks.items()}, name='bad_rows')
    .to_frame()
    .sort_values('bad_rows', ascending=False)
)
quality

,bad_rows
cancelled order still has a delivery time or rating,0
delivered order missing a rating,0
delivered order missing a delivery time,0
order_total is not positive,0
rating outside the 1-5 scale,0
status outside the known vocabulary,0
duplicate order_id,0
order references a customer that does not exist,0
order references a restaurant that does not exist,0
order references a courier that does not exist,0


## 7 — Reconciliation against the SQL

Two engines, one source of truth. DuckDB reads the same four CSVs directly — not
the pandas frames — and answers all twenty questions in SQL; each result is then
compared against the pandas value computed above.

A mirrored analysis is only worth anything if the mirror is checked, and this is
where the checking happens: a divergence here means one of the two idioms has a
bug, and both get read again until they agree.

In [26]:
import duckdb

con = duckdb.connect()
for table in ('couriers', 'customers', 'restaurants', 'orders'):
    con.execute(
        f"CREATE TABLE {table} AS "
        f"SELECT * FROM read_csv_auto('{(DATA_DIR / f'{table}.csv').as_posix()}')"
    )

con.execute('SELECT COUNT(*) FROM orders').fetchone()[0]

420

In [27]:
CHECKS = {
    'Q1  delivered GMV': (
        "SELECT SUM(order_total) FROM orders WHERE status = 'delivered'",
        gmv),
    'Q2  delivered orders': (
        "SELECT COUNT(*) FROM orders WHERE status = 'delivered'",
        len(delivered)),
    'Q3  average order value': (
        "SELECT AVG(order_total) FROM orders WHERE status = 'delivered'",
        delivered['order_total'].mean()),
    'Q4  Chinese cuisine revenue': (
        "SELECT SUM(o.order_total) FROM orders o JOIN restaurants r USING (restaurant_id) "
        "WHERE o.status = 'delivered' AND r.cuisine = 'Chinese'",
        revenue_by_cuisine.loc['Chinese', 'revenue']),
    'Q5  top restaurant revenue': (
        "SELECT SUM(o.order_total) AS revenue FROM orders o JOIN restaurants r USING (restaurant_id) "
        "WHERE o.status = 'delivered' GROUP BY r.restaurant_id ORDER BY revenue DESC LIMIT 1",
        top_restaurants['revenue'].iloc[0]),
    'Q6  cancellation rate %': (
        "SELECT 100.0 * COUNT(*) FILTER (WHERE status = 'cancelled') / COUNT(*) FROM orders",
        100 * cancelled.mean()),
    'Q7  New York revenue share %': (
        "SELECT ROUND(100.0 * SUM(o.order_total) / (SELECT SUM(order_total) FROM orders "
        "WHERE status = 'delivered'), 1) FROM orders o JOIN customers c USING (customer_id) "
        "WHERE o.status = 'delivered' AND c.city = 'New York'",
        city_revenue.loc['New York', 'pct_of_revenue']),
    'Q8  ARPU': (
        "SELECT SUM(order_total) / COUNT(DISTINCT customer_id) FROM orders WHERE status = 'delivered'",
        arpu),
    'Q9  repeat-customer rate %': (
        "WITH per_customer AS (SELECT customer_id, COUNT(*) AS n FROM orders "
        "WHERE status = 'delivered' GROUP BY customer_id) "
        "SELECT 100.0 * COUNT(*) FILTER (WHERE n > 1) / COUNT(*) FROM per_customer",
        100 * repeat.mean()),
    'Q10 customers never served': (
        "SELECT COUNT(*) FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o "
        "WHERE o.customer_id = c.customer_id AND o.status = 'delivered')",
        never_ordered),
    'Q11 March MoM change %': (
        "WITH monthly AS (SELECT EXTRACT(month FROM order_date) AS m, SUM(order_total) AS revenue "
        "FROM orders WHERE status = 'delivered' GROUP BY 1) "
        "SELECT mom FROM (SELECT m, 100.0 * (revenue - LAG(revenue) OVER (ORDER BY m)) "
        "/ LAG(revenue) OVER (ORDER BY m) AS mom FROM monthly) t WHERE m = 3",
        monthly.loc['2025-03', 'mom_pct']),
    'Q12 cumulative revenue to March': (
        "WITH monthly AS (SELECT EXTRACT(month FROM order_date) AS m, SUM(order_total) AS revenue "
        "FROM orders WHERE status = 'delivered' GROUP BY 1) "
        "SELECT cum FROM (SELECT m, SUM(revenue) OVER (ORDER BY m ROWS BETWEEN UNBOUNDED PRECEDING "
        "AND CURRENT ROW) AS cum FROM monthly) t WHERE m = 3",
        monthly.loc['2025-03', 'cumulative_revenue']),
    'Q13 slowest city, avg minutes': (
        "SELECT AVG(o.delivery_minutes) AS m FROM orders o JOIN customers c USING (customer_id) "
        "WHERE o.status = 'delivered' GROUP BY c.city ORDER BY m DESC LIMIT 1",
        delivery_by_city['avg_delivery_minutes'].max()),
    'Q14 Indian avg order rating': (
        "SELECT AVG(o.rating) FROM orders o JOIN restaurants r USING (restaurant_id) "
        "WHERE o.status = 'delivered' AND o.rating IS NOT NULL AND r.cuisine = 'Indian'",
        rating_by_cuisine.loc['Indian', 'avg_rating']),
    'Q15 top customer spend': (
        "SELECT SUM(o.order_total) AS spend FROM orders o JOIN customers c USING (customer_id) "
        "WHERE o.status = 'delivered' GROUP BY c.customer_id ORDER BY spend DESC LIMIT 1",
        top_customers['spend'].iloc[0]),
    'Q16 rank of Austin by revenue': (
        "WITH city_revenue AS (SELECT c.city, SUM(o.order_total) AS revenue FROM orders o "
        "JOIN customers c USING (customer_id) WHERE o.status = 'delivered' GROUP BY c.city) "
        "SELECT revenue_rank FROM (SELECT city, RANK() OVER (ORDER BY revenue DESC) AS revenue_rank "
        "FROM city_revenue) t WHERE city = 'Austin'",
        city_rank.loc['Austin', 'revenue_rank']),
    'Q17 orders above own AOV': (
        "WITH flagged AS (SELECT order_total, AVG(order_total) OVER (PARTITION BY customer_id) AS aov "
        "FROM orders WHERE status = 'delivered') "
        "SELECT COUNT(*) FILTER (WHERE order_total > aov) FROM flagged",
        above_own_aov.sum()),
    'Q18 largest signup cohort': (
        "SELECT COUNT(*) AS n FROM customers GROUP BY DATE_TRUNC('month', signup_date) "
        "ORDER BY n DESC LIMIT 1",
        cohorts['customers'].max()),
    'Q19 busiest courier deliveries': (
        "SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered' "
        "GROUP BY courier_id ORDER BY n DESC LIMIT 1",
        courier_stats['deliveries'].max()),
    'Q20 data-quality violations': (
        "SELECT (SELECT COUNT(*) FROM orders WHERE status = 'cancelled' "
        "AND (delivery_minutes IS NOT NULL OR rating IS NOT NULL)) "
        "+ (SELECT COUNT(*) FROM orders WHERE status = 'delivered' AND rating IS NULL) "
        "+ (SELECT COUNT(*) FROM orders WHERE order_total <= 0)",
        quality['bad_rows'].sum()),
}

rows = []
for label, (query, pandas_value) in CHECKS.items():
    sql_value = round(float(con.execute(query).fetchone()[0]), 2)
    pandas_value = round(float(pandas_value), 2)
    rows.append({'check': label, 'sql': sql_value, 'pandas': pandas_value,
                 'match': sql_value == pandas_value})

reconciliation = pd.DataFrame(rows).set_index('check')
print(f"{reconciliation['match'].sum()} / {len(reconciliation)} answers reconcile")
reconciliation

20 / 20 answers reconcile

,sql,pandas,match
check,,,
Q1 delivered GMV,"16,413.67","16,413.67",True
Q2 delivered orders,341.00,341.00,True
Q3 average order value,48.13,48.13,True
Q4 Chinese cuisine revenue,"5,365.53","5,365.53",True
Q5 top restaurant revenue,"1,455.54","1,455.54",True
Q6 cancellation rate %,18.81,18.81,True
Q7 New York revenue share %,30.80,30.80,True
Q8 ARPU,145.25,145.25,True
Q9 repeat-customer rate %,82.30,82.30,True


In [28]:
assert reconciliation['match'].all(), reconciliation[~reconciliation['match']]
print('All 20 questions agree between PostgreSQL-style SQL and pandas.')

All 20 questions agree between PostgreSQL-style SQL and pandas.


## Takeaways

* **16,413.67 GMV** across **341 delivered** of 420 orders — an **18.81%**
  cancellation rate that costs roughly a fifth of demand before it converts.
* **Chinese, Thai and Italian cuisines carry 79% of revenue** (5,365.53 / 4,126.08
  / 3,453.90); American, Mexican and Indian sit near 1,150 each.
* **New York is 30.8% of revenue**, more than Seattle and Miami combined, but it
  is also the second-slowest city at 45.48 average delivery minutes.
* **113 of 120 registered customers are active**, ARPU is **145.25**, and **82.3%
  of active customers reorder** — retention, not acquisition, is what this
  marketplace runs on.
* **Ratings are tight (3.79–4.16 across cuisines)** while courier averages spread
  wider (3.80–4.26) at near-identical delivery volumes — the variance that matters
  is on the courier side, not the kitchen side.
* **All 20 answers reconcile exactly between SQL and pandas**, and all ten
  data-quality invariants hold at zero violations.